# Pratica 11 — Ricerca e recupero classico

**Corso:** NLP Classico con Python · **Lezione 11** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase11_busqueda_recuperacion_clasica_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- I 5 documenti di `DOCS_P` sono minuscoli e pensati per poter essere calcolati a mano (indicato in ogni esercizio).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas`.

---
## Cella 0 — Setup e dati

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
print("Librerie pronte.")

In [ ]:
import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)                        # seme fisso (anche se questo argomento non addestra nulla di casuale)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

try:
    import nltk
    from nltk.corpus import stopwords
    stopwords.words("spanish")
except LookupError:
    nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = {"es": set(stopwords.words("spanish")), "it": set(stopwords.words("italian"))}   # dato-es

def tokenizar(texto, idioma, quitar_stopwords=True):
    """Minuscole, solo lettere (senza numeri né punteggiatura), e stopword opzionali."""
    palabras = re.findall(r"[^\W\d_]+", texto.lower())
    if quitar_stopwords:
        palabras = [p for p in palabras if p not in STOPWORDS[idioma]]
    return palabras

def construir_vocabulario(documentos_tok):
    """Lista ordinata di parole distinte che compaiono in almeno un documento."""
    vocab = sorted({p for doc in documentos_tok for p in doc})
    return vocab, {p: i for i, p in enumerate(vocab)}

def matriz_tf(documentos_tok, indice):
    """Matrice di conteggi (documenti x vocabolario): TF grezzo, non normalizzato."""
    M = np.zeros((len(documentos_tok), len(indice)), dtype=float)
    for i, doc in enumerate(documentos_tok):
        for palabra, n in Counter(doc).items():
            if palabra in indice:
                M[i, indice[palabra]] = n
    return M

def matriz_tfidf(TF, idf):
    """TF-IDF = TF * IDF, con ogni riga normalizzata a norma 1 (così il coseno è un prodotto scalare)."""
    X = TF * idf
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    normas[normas == 0] = 1.0
    return X / normas

def dividir_frases(texto):
    """Taglia un testo in frasi per . ! ? e scarta quelle vuote."""
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto.strip()) if f.strip()]

print("Funzioni ausiliarie caricate.")

In [ ]:
# datos-es
# Cinque documenti minuscoli (identici in ES e IT: sono dati) per gli esercizi 1-4
DOCS_P = ["la casa tiene un jardín grande", "el jardín de la casa tiene flores", "el coche está en el garaje",
          "el coche nuevo es rápido", "la bicicleta está en el jardín"]

In [ ]:
tok_p = [d.split() for d in DOCS_P]
vocab_p, idx_p = construir_vocabulario(tok_p)
TF_p = matriz_tf(tok_p, idx_p)
print("vocabolario:", vocab_p)
print("df di 'jardín':", int((TF_p[:, idx_p["jardín"]] > 0).sum()), "su 5 documenti")   # dato-es

## Esercizio 1 — IDF del TF-IDF classico

Completa `idf_tfidf_p(TF)`: `log((1+N)/(1+df)) + 1`, con `N` = numero di documenti (righe di `TF`) e `df` il numero di documenti in cui compare ogni parola (colonna con almeno un valore > 0). Con `DOCS_P` (5 documenti), `jardín` compare in 3 di essi: a mano, `idf = log(6/4) + 1 = 0,405 + 1 = 1,405`.

In [ ]:
def idf_tfidf_p(TF):
    return None    # TODO: log((1+N)/(1+df)) + 1, con N = TF.shape[0] e df = (TF > 0).sum(axis=0)

# Controllo automatico
_w_jardin, _w_garaje = "jardín", "garaje"   # dato-es
_idf = idf_tfidf_p(TF_p)
assert _idf is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_idf[idx_p[_w_jardin]], np.log(6 / 4) + 1), "a mano: log(6/4)+1 ≈ 1,405"
assert (_idf >= 1.0).all(), "l'idf del TF-IDF classico non scende mai sotto 1"
assert np.isclose(_idf[idx_p[_w_garaje]], np.log(6 / 2) + 1)   # compare in 1 documento
print("Esercizio 1 corretto ✔", round(float(_idf[idx_p[_w_jardin]]), 3))

## Esercizio 2 — Vettore TF-IDF di una query e ricerca per coseno

Completa `buscar_coseno_p(consulta_tok, X, indice, idf, k)`: costruisci il vettore TF-IDF della query (TF della query moltiplicato per l'`idf` dei documenti, senza dimenticare di normalizzare a norma 1 se la norma non è 0) e restituisci i `k` documenti di `X` (già normalizzato per riga) con il prodotto scalare più alto, insieme ai loro punteggi. Usa `Counter` per il TF della query.

In [ ]:
def buscar_coseno_p(consulta_tok, X, indice, idf, k=3):
    v = np.zeros(len(indice))
    # TODO: riempi v[indice[palabra]] = conteggio della parola in consulta_tok, per le parole presenti in indice
    v = v * idf
    return None, None    # TODO: normalizza v (se la sua norma > 0), calcola X @ v, e restituisci gli indici e i punteggi dei k documenti con punteggio più alto (np.argsort)

# Controllo automatico
_idf_p = idf_tfidf_p(TF_p)
_X_p = matriz_tfidf(TF_p, _idf_p)
_orden, _punt = buscar_coseno_p(["coche", "garaje"], _X_p, idx_p, _idf_p, k=5)
assert _orden is not None, "Completa le righe contrassegnate con TODO"
assert list(_orden[:2]) == [2, 3], "i documenti 2 e 3 (indice 0) parlano di auto/garage; dovrebbero venire prima"
assert _punt[0] > _punt[-1] and np.isclose(np.linalg.norm(_X_p[0]), 1.0)
_o2, _p2 = buscar_coseno_p(["palabra_inexistente"], _X_p, idx_p, _idf_p, k=5)
assert np.allclose(_p2, 0.0)   # query senza parole del vocabolario: punteggio 0 per tutti
print("Esercizio 2 corretto ✔", list(_orden))

## Esercizio 3 — IDF probabilistico di BM25 (e la sua correzione)

Completa `idf_bm25_p(TF, corregido=False)`: senza correzione, `log((N-df+0,5)/(df+0,5))`; con `corregido=True`, `log(1 + (N-df+0,5)/(df+0,5))`. Con `DOCS_P`, la parola `el` compare in 4 dei 5 documenti (più della metà): a mano, `idf_bm25(el) = log((5-4+0,5)/(4+0,5)) = log(1,5/4,5) = log(0,333) ≈ -1,099` (negativo).

In [ ]:
def idf_bm25_p(TF, corregido=False):
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    razon = None    # TODO: (N - df + 0.5) / (df + 0.5)
    return razon

# Controllo automatico
_w_el = "el"   # dato-es
_ib = idf_bm25_p(TF_p)
assert _ib is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_ib[idx_p[_w_el]], np.log(1.5 / 4.5)), "a mano: log(1,5/4,5) ≈ -1,099"
assert _ib[idx_p[_w_el]] < 0, "è in più della metà dei documenti: idf negativo"
_ib_c = idf_bm25_p(TF_p, corregido=True)
assert (_ib_c >= 0).all() and _ib_c[idx_p[_w_el]] > 0, "l'idf corretto non è mai negativo"
print("Esercizio 3 corretto ✔", "idf_bm25 =", round(float(_ib[idx_p[_w_el]]), 3), "| corregido =", round(float(_ib_c[idx_p[_w_el]]), 3))

## Esercizio 4 — Precisione@k e MRR

Completa `precision_en_k_p(recuperados, relevantes, k)` (frazione di rilevanti tra i primi `k` di `recuperados`) e `rango_reciproco_p(recuperados, relevantes)` (`1/posizione` del primo rilevante, indicizzata da 1; `0.0` se nessuno dei `recuperados` è rilevante). A mano: `recuperados = [5, 2, 9, 1]`, `relevantes = {9, 1}`, `k=3` → tra `[5, 2, 9]` c'è 1 rilevante (`9`): `precisione@3 = 1/3`. Il primo rilevante (`9`) è in posizione 3: `rango_reciproco = 1/3`.

In [ ]:
def precision_en_k_p(recuperados, relevantes, k):
    top = recuperados[:k]
    return None    # TODO: numero di elementi di top presenti in relevantes, diviso per k

def rango_reciproco_p(recuperados, relevantes):
    pass    # TODO: scorri recuperados con enumerate(recuperados, 1); se il documento è in relevantes, restituisci 1/posizione
    return 0.0

# Controllo automatico
_p3 = precision_en_k_p([5, 2, 9, 1], {9, 1}, 3)
assert _p3 is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_p3, 1 / 3), "a mano: 1 su 3 = 1/3"
assert np.isclose(precision_en_k_p([9, 1, 5, 2], {9, 1}, 2), 1.0)
assert np.isclose(precision_en_k_p([5, 2, 3], {9, 1}, 3), 0.0)
_rr = rango_reciproco_p([5, 2, 9, 1], {9, 1})
assert np.isclose(_rr, 1 / 3), "il primo rilevante (9) è in posizione 3"
assert np.isclose(rango_reciproco_p([9, 5, 2], {9, 1}), 1.0)
assert np.isclose(rango_reciproco_p([5, 2, 3], {9, 1}), 0.0)
print("Esercizio 4 corretto ✔", "precisione@3 =", round(_p3, 3), "| MRR di un esempio =", round(_rr, 3))

## Esercizio 5 — Parole chiave con TF-IDF

Completa `palabras_clave_p(texto, idf_global, indice_global, n=3)`: tokenizza `texto` (usa `tokenizar(texto, "es")`), conta le parole con `Counter`, pesa ognuna con `conteggio * idf_global[indice_global[parola]]` (solo le parole presenti in `indice_global`) e restituisci le `n` parole con punteggio più alto, dalla più alta alla più bassa.

In [ ]:
def palabras_clave_p(texto, idf_global, indice_global, n=3):
    tok = tokenizar(texto, "es")   # dato-es
    cuenta = Counter(tok)
    puntuadas = []    # TODO: lista di (parola, conteggio * idf_global[indice_global[parola]]) per le parole di cuenta presenti in indice_global
    puntuadas.sort(key=lambda par: -par[1])
    return [p for p, _ in puntuadas[:n]]

# Controllo automatico
_idf_p3 = idf_tfidf_p(TF_p)
_texto_prueba = "el coche nuevo está en el garaje"   # dato-es
_claves = palabras_clave_p(_texto_prueba, _idf_p3, idx_p, n=3)
assert _claves, "Completa le righe contrassegnate con TODO"
assert "garaje" in _claves, "'garaje' compare solo in 1 dei 5 documenti: dovrebbe avere un idf alto ed essere tra le parole chiave"
_w_el2 = "el"   # dato-es
assert _w_el2 not in _claves, "compare in più della metà dei documenti: idf basso, non dovrebbe essere parola chiave"
print("Esercizio 5 corretto ✔", _claves)

---
## Sfida finale — Riassunto estrattivo per punteggio di frase

Con `idf_tfidf_p`, `construir_vocabulario`, `matriz_tf` e `tokenizar` (già disponibili) e `dividir_frases` (già disponibile):

1. Completa `resumen_extractivo_p(texto, idf_global, indice_global, n_frases=1)`: dividi `texto` in frasi (`dividir_frases`), pesa ogni frase con la **media** di `conteggio_nella_frase * idf_global[indice_global[parola]]` sulle parole della frase presenti in `indice_global` (se nessuna lo è, il punteggio è 0,0), e restituisci le `n_frases` con punteggio più alto **nel loro ordine originale** nel testo (non per punteggio).
2. Applicalo al testo `TEXTO_RETO` (definito sotto, 3 frasi) usando l'IDF di `DOCS_P` e salva il risultato in `resumen_reto`.
3. Verifica che la frase scelta sia quella che menziona `jardín` e `flores`: le altre due frasi ripetono `casa` e `coche`, parole che compaiono anch'esse in più di un documento di `DOCS_P` e non bastano per vincere.

Richiede meno di un secondo.

In [ ]:
TEXTO_RETO = "El coche está en la casa. El jardín de la casa tiene flores. La casa tiene un coche."   # dato-es

def resumen_extractivo_p(texto, idf_global, indice_global, n_frases=1):
    frases = dividir_frases(texto)
    puntuaciones = []
    for frase in frases:
        pass    # TODO: tokenizza la frase con tokenizar (la lingua è lo spagnolo), calcola la media di conteggio*idf_global[...] sulle parole in indice_global (0.0 se nessuna lo è) e aggiungila a puntuaciones
    mejores = sorted(range(len(frases)), key=lambda i: -puntuaciones[i])[:n_frases]
    mejores.sort()
    return [frases[i] for i in mejores]

resumen_reto = None   # TODO: resumen_extractivo_p(TEXTO_RETO, idf_tfidf_p(TF_p), idx_p, n_frases=1)

# Controllo automatico
_w_jardin2, _w_flores = "jardín", "flores"   # dato-es
assert resumen_reto is not None, "Completa le righe contrassegnate con TODO"
assert len(resumen_reto) == 1 and _w_jardin2 in resumen_reto[0].lower() and _w_flores in resumen_reto[0].lower(), "dovrebbe scegliere la frase del giardino con i fiori"
print("Sfida finale corretta ✔", resumen_reto)

**Le tue conclusioni (scrivi 3-4 frasi):** perché l'IDF del TF-IDF classico non è mai negativo, mentre quello di BM25 sì può esserlo? Cosa significa che la precisione@3 di una query con un unico documento rilevante non può superare 1/3? Perché una parola chiave ha bisogno di un IDF alto, non solo di un TF alto? Cosa hanno in comune l'estrazione di parole chiave e il riassunto estrattivo?

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def idf_tfidf_p(TF):
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    return np.log((1 + N) / (1 + df)) + 1.0

# Controllo automatico
_w_jardin, _w_garaje = "jardín", "garaje"   # dato-es
_idf = idf_tfidf_p(TF_p)
assert _idf is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_idf[idx_p[_w_jardin]], np.log(6 / 4) + 1), "a mano: log(6/4)+1 ≈ 1,405"
assert (_idf >= 1.0).all(), "l'idf del TF-IDF classico non scende mai sotto 1"
assert np.isclose(_idf[idx_p[_w_garaje]], np.log(6 / 2) + 1)   # compare in 1 documento
print("Esercizio 1 corretto ✔", round(float(_idf[idx_p[_w_jardin]]), 3))

### Soluzione dell'esercizio 2

In [ ]:
def buscar_coseno_p(consulta_tok, X, indice, idf, k=3):
    v = np.zeros(len(indice))
    for palabra, n in Counter(consulta_tok).items():
        if palabra in indice:
            v[indice[palabra]] = n
    v = v * idf
    norma = np.linalg.norm(v)
    if norma > 0:
        v = v / norma
    puntuaciones = X @ v
    orden = np.argsort(-puntuaciones)[:k]
    return orden, puntuaciones[orden]

# Controllo automatico
_idf_p = idf_tfidf_p(TF_p)
_X_p = matriz_tfidf(TF_p, _idf_p)
_orden, _punt = buscar_coseno_p(["coche", "garaje"], _X_p, idx_p, _idf_p, k=5)
assert _orden is not None, "Completa le righe contrassegnate con TODO"
assert list(_orden[:2]) == [2, 3], "i documenti 2 e 3 (indice 0) parlano di auto/garage; dovrebbero venire prima"
assert _punt[0] > _punt[-1] and np.isclose(np.linalg.norm(_X_p[0]), 1.0)
_o2, _p2 = buscar_coseno_p(["palabra_inexistente"], _X_p, idx_p, _idf_p, k=5)
assert np.allclose(_p2, 0.0)   # query senza parole del vocabolario: punteggio 0 per tutti
print("Esercizio 2 corretto ✔", list(_orden))

### Soluzione dell'esercizio 3

In [ ]:
def idf_bm25_p(TF, corregido=False):
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    razon = (N - df + 0.5) / (df + 0.5)
    return np.log(1.0 + razon) if corregido else np.log(razon)

# Controllo automatico
_w_el = "el"   # dato-es
_ib = idf_bm25_p(TF_p)
assert _ib is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_ib[idx_p[_w_el]], np.log(1.5 / 4.5)), "a mano: log(1,5/4,5) ≈ -1,099"
assert _ib[idx_p[_w_el]] < 0, "è in più della metà dei documenti: idf negativo"
_ib_c = idf_bm25_p(TF_p, corregido=True)
assert (_ib_c >= 0).all() and _ib_c[idx_p[_w_el]] > 0, "l'idf corretto non è mai negativo"
print("Esercizio 3 corretto ✔", "idf_bm25 =", round(float(_ib[idx_p[_w_el]]), 3), "| corregido =", round(float(_ib_c[idx_p[_w_el]]), 3))

### Soluzione dell'esercizio 4

In [ ]:
def precision_en_k_p(recuperados, relevantes, k):
    top = recuperados[:k]
    return len([d for d in top if d in relevantes]) / k

def rango_reciproco_p(recuperados, relevantes):
    for i, d in enumerate(recuperados, 1):
        if d in relevantes:
            return 1.0 / i
    return 0.0

# Controllo automatico
_p3 = precision_en_k_p([5, 2, 9, 1], {9, 1}, 3)
assert _p3 is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_p3, 1 / 3), "a mano: 1 su 3 = 1/3"
assert np.isclose(precision_en_k_p([9, 1, 5, 2], {9, 1}, 2), 1.0)
assert np.isclose(precision_en_k_p([5, 2, 3], {9, 1}, 3), 0.0)
_rr = rango_reciproco_p([5, 2, 9, 1], {9, 1})
assert np.isclose(_rr, 1 / 3), "il primo rilevante (9) è in posizione 3"
assert np.isclose(rango_reciproco_p([9, 5, 2], {9, 1}), 1.0)
assert np.isclose(rango_reciproco_p([5, 2, 3], {9, 1}), 0.0)
print("Esercizio 4 corretto ✔", "precisione@3 =", round(_p3, 3), "| MRR di un esempio =", round(_rr, 3))

### Soluzione dell'esercizio 5

In [ ]:
def palabras_clave_p(texto, idf_global, indice_global, n=3):
    tok = tokenizar(texto, "es")   # dato-es
    cuenta = Counter(tok)
    puntuadas = [(p, n_ * idf_global[indice_global[p]]) for p, n_ in cuenta.items() if p in indice_global]
    puntuadas.sort(key=lambda par: -par[1])
    return [p for p, _ in puntuadas[:n]]

# Controllo automatico
_idf_p3 = idf_tfidf_p(TF_p)
_texto_prueba = "el coche nuevo está en el garaje"   # dato-es
_claves = palabras_clave_p(_texto_prueba, _idf_p3, idx_p, n=3)
assert _claves, "Completa le righe contrassegnate con TODO"
assert "garaje" in _claves, "'garaje' compare solo in 1 dei 5 documenti: dovrebbe avere un idf alto ed essere tra le parole chiave"
_w_el2 = "el"   # dato-es
assert _w_el2 not in _claves, "compare in più della metà dei documenti: idf basso, non dovrebbe essere parola chiave"
print("Esercizio 5 corretto ✔", _claves)

### Soluzione della sfida finale

In [ ]:
TEXTO_RETO = "El coche está en la casa. El jardín de la casa tiene flores. La casa tiene un coche."   # dato-es

def resumen_extractivo_p(texto, idf_global, indice_global, n_frases=1):
    frases = dividir_frases(texto)
    puntuaciones = []
    for frase in frases:
        tok = tokenizar(frase, "es")   # dato-es
        cuenta = Counter(tok)
        pesos = [cuenta[p] * idf_global[indice_global[p]] for p in cuenta if p in indice_global]
        puntuaciones.append(float(np.mean(pesos)) if pesos else 0.0)
    mejores = sorted(range(len(frases)), key=lambda i: -puntuaciones[i])[:n_frases]
    mejores.sort()
    return [frases[i] for i in mejores]

resumen_reto = resumen_extractivo_p(TEXTO_RETO, idf_tfidf_p(TF_p), idx_p, n_frases=1)

# Controllo automatico
_w_jardin2, _w_flores = "jardín", "flores"   # dato-es
assert resumen_reto is not None, "Completa le righe contrassegnate con TODO"
assert len(resumen_reto) == 1 and _w_jardin2 in resumen_reto[0].lower() and _w_flores in resumen_reto[0].lower(), "dovrebbe scegliere la frase del giardino con i fiori"
print("Sfida finale corretta ✔", resumen_reto)

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (TF-IDF contro BM25, l'IDF negativo, precisione@k contro MRR, parole chiave, riassunto estrattivo)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*